# GenAI-Net (RL4CRN) Tutorial 02 — Deterministic Tracking Task

This notebook is a **compact** end-to-end tutorial for the deterministic **tracking** task.

Goal: learn a CRN that makes the output track a target signal derived from the inputs (e.g., copy input0).


---
## 0) Environment sanity check


In [ ]:
import os, sys, numpy as np

print("Python:", sys.version.split()[0])
print("CWD:", os.getcwd())


---
## 1) Imports

We use the standard interface layer:
- `Configurator`, `make_task`, `make_session_and_trainer`


In [ ]:
import numpy as np

from RL4CRN.utils.input_interface import (
    Configurator,
    make_task,
    make_session_and_trainer,
)


from RL4CRN.utils.visualizations import plot_truth_table  # optional; may not be used here

---
## 2) Notebook-local helpers


In [ ]:
def print_task_summary(task, max_preview=3):
    print("Task:", task.name)
    print("time_horizon:", task.time_horizon.shape, f"[0..{task.time_horizon[-1]}]")
    print("num scenarios:", len(task.u_list))
    if len(task.u_list) > 0:
        print(f"first {min(max_preview, len(task.u_list))} u:", task.u_list[:max_preview])
    print()

def run_smoke_reward(task, state, label=""):
    out = task.compute_reward(state)
    if isinstance(out, tuple):
        loss, info = out
    else:
        loss, info = out, {}
    print(f"[reward smoke{(' - ' + label) if label else ''}] loss={float(loss):.6g} | info_keys={list(info.keys())[:10]}")
    return out


---
## 3) Define the task (standalone)

Key knobs:
- `p`: parameterization / problem setting size (task dependent)
- `u_values`: input grid (scenarios)
- `target`: what to track (e.g. `"copy_input0"`)
- `weights="transient"`: emphasize trajectory matching over time


In [ ]:
species_labels = ["X_1","X_2","X_3","X_4","X_5","X_6","OUT"]

task = make_task(
    kind="tracking",
    species_labels=species_labels,
    p=3,
    u_values=[0.5, 1.0, 1.5],
    target="copy_input0",
    ic=("constant", 0.01),
    weights="transient",
    t_f=50, n_t=120,
)

print_task_summary(task)


---
## 5) Full wiring + training loop (compact)

Pattern:
1. Configure `cfg`
2. Build `session, trainer`
3. Smoke-test reward on the template
4. Run a short training
5. Inspect the best


In [ ]:
cfg = Configurator.preset("fast")

# ---- Task ----
cfg.task.kind = "tracking"
cfg.task.n_inputs = 3
cfg.task.p = 3
cfg.task.u_values = [0.5, 1.0, 1.5]
cfg.task.target = "copy_input0"
cfg.task.weights = "transient"
cfg.task.t_f = 50.0
cfg.task.N_t = 120
cfg.task.ic_value = 0.01

# ---- Train ----
cfg.train.max_added_reactions = 4
cfg.train.epochs = 8
cfg.train.render_every = 2
cfg.train.seed = 1

session, trainer = make_session_and_trainer(cfg, device="auto")
print_task_summary(session.task)
run_smoke_reward(session.task, session.crn_template, label="template")

trainer.run(epochs=cfg.train.epochs, checkpoint_path=None)
trainer.inspect_best(plot=True)


---
## 6) Checkpoint roundtrip (optional)

For longer runs, always checkpoint. This cell demonstrates a minimal resume workflow.


In [ ]:
# Uncomment for longer runs:
# checkpoint_path = "tracking_chkpt.pkl"
# trainer.run(epochs=cfg.train.epochs, checkpoint_path=checkpoint_path)
# session2, trainer2 = make_session_and_trainer(cfg, device="auto")
# trainer2.load(checkpoint_path)
# trainer2.inspect_best(plot=True)


---
## 7) Customize

Common customizations:
- change `target` (if supported) or pass a custom target generator (if your API supports it)
- increase `u_values` density / range
- switch `weights` between `"transient"` and `"steady_state"`
- adjust `t_f`, `N_t` for faster iteration vs better resolution
